In [1]:
"""
test_distreebu_rs.py
====================
Correctness + speed comparison between the Rust-backed distreebu_rs package
and the original pure-Python DisTreebution package.

Usage
-----
# With both packages available:
    python test_distreebu_rs.py

# Rust-only (skip Python comparison):
    python test_distreebu_rs.py --rust-only

Requirements
------------
    pip install distreebu_rs-*.whl numpy
    # optional (for comparison): DisTreebution folder on sys.path
"""

import argparse
import sys
import time
import textwrap
from pathlib import Path

import numpy as np

# ── colour helpers ─────────────────────────────────────────────────────────────
GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
CYAN   = "\033[96m"
BOLD   = "\033[1m"
DIM    = "\033[2m"
RESET  = "\033[0m"

def ok(msg):    print(f"  {GREEN}✓{RESET} {msg}")
def fail(msg):  print(f"  {RED}✗{RESET} {msg}"); _FAILURES.append(msg)
def info(msg):  print(f"  {DIM}{msg}{RESET}")
def header(msg):
    print(f"\n{BOLD}{CYAN}{'─'*60}{RESET}")
    print(f"{BOLD}{CYAN}  {msg}{RESET}")
    print(f"{BOLD}{CYAN}{'─'*60}{RESET}")

_FAILURES = []

# ── timing helper ──────────────────────────────────────────────────────────────
def timeit(fn, n=10):
    """Return mean wall-clock time in ms over n calls."""
    # one warm-up
    fn()
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n * 1e3


def speedup_str(py_ms, rs_ms):
    ratio = py_ms / rs_ms
    colour = GREEN if ratio >= 5 else (YELLOW if ratio >= 2 else RED)
    return f"{colour}{ratio:.1f}×{RESET}"


def print_speed_row(label, py_ms, rs_ms):
    bar_len = 30
    ratio = py_ms / rs_ms
    rs_bar = max(1, round(bar_len / ratio))
    py_bar = bar_len
    print(
        f"  {label:<36s}"
        f"  Py {py_ms:>7.1f} ms  {'█'*py_bar}"
    )
    print(
        f"  {'':36s}"
        f"  Rs {rs_ms:>7.1f} ms  {'█'*rs_bar}  {speedup_str(py_ms, rs_ms)}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# 1. Import packages
# ══════════════════════════════════════════════════════════════════════════════
header("Importing packages")

# Rust package
try:
    import distreebu_rs as rs
    ok(f"distreebu_rs imported  (symbols: {', '.join(s for s in dir(rs) if not s.startswith('_'))})")
    HAS_RUST = True
except ImportError as e:
    fail(f"distreebu_rs not found: {e}")
    HAS_RUST = False

# Python package (optional)
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument("--rust-only", action="store_true")
parser.add_argument("--py-path", default=".", help="Path to folder containing DisTreebution/")
args, _ = parser.parse_known_args()

HAS_PYTHON = False
if not args.rust_only:
    if args.py_path not in sys.path:
        sys.path.insert(0, args.py_path)
    try:
        from DisTreebution.RT.entropies_Quadratic      import entropies_Quadratic      as py_ent_quad
        from DisTreebution.CRPSRT.entropies_CRPS        import entropies_CRPS            as py_ent_crps
        from DisTreebution.QRT.entropies_MultiQuantiles import entropies_MultiQuantiles  as py_ent_mq
        from DisTreebution.RT.RegressionTreeQuadratic   import RegressionTreeQuadratic   as PyRTQ
        from DisTreebution.CRPSRT.RegressionTree        import RegressionTree            as PyCRPS
        from DisTreebution.QRT.RegressionTreeQuantile   import RegressionTreeQuantile    as PyQRT
        ok("DisTreebution (Python) imported")
        HAS_PYTHON = True
    except ImportError as e:
        print(f"  {YELLOW}⚠{RESET}  DisTreebution not found ({e}) — running Rust-only checks")
        print(f"  {DIM}Tip: python test_distreebu_rs.py --py-path /path/to/folder/containing/DisTreebution/{RESET}")

if not HAS_RUST:
    print(f"\n{RED}Cannot continue without distreebu_rs.{RESET}")
    sys.exit(1)


# ══════════════════════════════════════════════════════════════════════════════
# 2. Data structures
# ══════════════════════════════════════════════════════════════════════════════
header("Data structures")

# ── FenwickTree ────────────────────────────────────────────────────────────────
print(f"\n{BOLD}FenwickTree{RESET}")
vals = [1.0, 3.0, 2.0, 5.0, 4.0]
ft = rs.FenwickTree(len(vals))
for i, v in enumerate(vals):
    ft.add(i, v)

checks = [
    ("prefix_sum(0) == 0",   abs(ft.prefix_sum(0)) < 1e-12),
    ("prefix_sum(1) == 1",   abs(ft.prefix_sum(1) - 1.0)  < 1e-12),
    ("prefix_sum(3) == 6",   abs(ft.prefix_sum(3) - 6.0)  < 1e-12),
    ("prefix_sum(5) == 15",  abs(ft.prefix_sum(5) - 15.0) < 1e-12),
    ("range_sum(1,3) == 5",  abs(ft.range_sum(1, 3) - 5.0) < 1e-12),
    ("range_sum(2,5) == 11", abs(ft.range_sum(2, 5) - 11.0)< 1e-12),
    ("__getitem__(0) == 1",  abs(ft[0] - 1.0) < 1e-12),
    ("__getitem__(3) == 5",  abs(ft[3] - 5.0) < 1e-12),
    ("len == 5",             len(ft) == 5),
]
for label, passed in checks:
    ok(label) if passed else fail(label)

# init helper
ft2 = rs.FenwickTree(5)
ft2.init([1.0, 3.0, 2.0, 5.0, 4.0])
ok("init() prefix_sum(5)==15") if abs(ft2.prefix_sum(5) - 15.0) < 1e-12 else fail("init() prefix_sum")

# ── MinMaxHeap ────────────────────────────────────────────────────────────────
print(f"\n{BOLD}MinMaxHeap{RESET}")
h = rs.MinMaxHeap()
for v in [5.0, 1.0, 3.0, 2.0, 4.0]:
    h.insert(v)

checks = [
    ("len == 5",         len(h) == 5),
    ("peekmin == 1.0",   h.peekmin() == 1.0),
    ("peekmax == 5.0",   h.peekmax() == 5.0),
    ("popmin == 1.0",    h.popmin()  == 1.0),
    ("peekmin == 2.0",   h.peekmin() == 2.0),
    ("popmax == 5.0",    h.popmax()  == 5.0),
    ("peekmax == 4.0",   h.peekmax() == 4.0),
    ("len == 3 after 2 pops", len(h) == 3),
]
for label, passed in checks:
    ok(label) if passed else fail(label)

# stress: sorted order via repeated popmin
h2 = rs.MinMaxHeap()
rng = np.random.default_rng(0)
vals_rand = rng.uniform(-10, 10, 100).tolist()
for v in vals_rand:
    h2.insert(v)
popped = []
while len(h2) > 0:
    popped.append(h2.popmin())
sorted_expected = sorted(vals_rand)
heap_sorted_ok = all(abs(a - b) < 1e-12 for a, b in zip(popped, sorted_expected))
ok("MinMaxHeap stress: popmin gives sorted order (n=100)") if heap_sorted_ok else fail("MinMaxHeap stress sort")

h3 = rs.MinMaxHeap()
for v in vals_rand:
    h3.insert(v)
popped_max = []
while len(h3) > 0:
    popped_max.append(h3.popmax())
heap_max_ok = all(abs(a - b) < 1e-12 for a, b in zip(popped_max, sorted(vals_rand, reverse=True)))
ok("MinMaxHeap stress: popmax gives reverse-sorted order (n=100)") if heap_max_ok else fail("MinMaxHeap stress max-sort")


# ══════════════════════════════════════════════════════════════════════════════
# 3. Entropy functions — correctness
# ══════════════════════════════════════════════════════════════════════════════
header("Entropy functions — correctness")

rng = np.random.default_rng(42)
SIZES = [10, 50, 200]
QUANTILES_LIST = [
    [0.5],
    [0.25, 0.75],
    [0.1, 0.25, 0.5, 0.75, 0.9],
]
ATOL = 1e-9

for n in SIZES:
    y     = rng.standard_normal(n)
    order = np.argsort(rng.standard_normal(n))
    o_lst = order.tolist()
    y_lst = y.tolist()

    # ── Quadratic ──────────────────────────────────────────────────────────────
    rs_q = np.array(rs.entropies_quadratic(o_lst, y_lst))
    label = f"entropies_quadratic  shape ({n+1},)"
    ok(label) if rs_q.shape == (n + 1,) else fail(label + f" got {rs_q.shape}")
    ok(f"  entropy[0]==0 and entropy[1]==0 for n={n}") if rs_q[0]==0 and rs_q[1]==0 else fail(f"  boundary zeros n={n}")
    ok(f"  all values >= 0 for n={n}") if np.all(rs_q >= -1e-12) else fail(f"  negative values n={n}")

    if HAS_PYTHON:
        py_q = py_ent_quad(order, y)
        diff = np.max(np.abs(py_q - rs_q))
        ok(f"  Rust==Python  max_diff={diff:.2e}  n={n}") if diff < ATOL else fail(f"  Rust!=Python diff={diff:.2e} n={n}")

    # ── CRPS ───────────────────────────────────────────────────────────────────
    rs_c = np.array(rs.entropies_crps(o_lst, y_lst))
    ok(f"entropies_crps       shape ({n+1},)") if rs_c.shape == (n + 1,) else fail(f"entropies_crps shape n={n}")
    ok(f"  entropy[0]==0 for n={n}") if rs_c[0] == 0 else fail(f"  entropy[0]!=0 n={n}")

    if HAS_PYTHON:
        py_c = py_ent_crps(order, y)
        diff = np.max(np.abs(py_c - rs_c))
        ok(f"  Rust==Python  max_diff={diff:.2e}  n={n}") if diff < ATOL else fail(f"  Rust!=Python diff={diff:.2e} n={n}")

    # ── MultiQuantiles ─────────────────────────────────────────────────────────
    for qs in QUANTILES_LIST:
        rs_mq = np.array(rs.entropies_multi_quantiles(o_lst, y_lst, qs))
        ok(f"entropies_multi_quantiles  q={qs}  shape ({n+1},)") \
            if rs_mq.shape == (n + 1,) else fail(f"entropies_multi_quantiles shape n={n} q={qs}")

        if HAS_PYTHON:
            py_mq = py_ent_mq(order, y, qs)
            diff = np.max(np.abs(py_mq - rs_mq))
            ok(f"  Rust==Python  max_diff={diff:.2e}  n={n}") if diff < ATOL else fail(f"  Rust!=Python diff={diff:.2e} n={n} q={qs}")

# LOO / bias-correction flag
for loo_flag in [True, False]:
    y_l  = rng.standard_normal(30).tolist()
    o_l  = list(np.argsort(rng.standard_normal(30)))
    r1 = rs.entropies_quadratic(o_l, y_l, loo_flag)
    r2 = rs.entropies_crps(o_l, y_l, loo_flag)
    r3 = rs.entropies_multi_quantiles(o_l, y_l, [0.25, 0.5, 0.75], loo_flag)
    ok(f"LOO={loo_flag}: all entropy functions return length-31 list") \
        if len(r1)==31 and len(r2)==31 and len(r3)==31 \
        else fail(f"LOO={loo_flag} length error")


# ══════════════════════════════════════════════════════════════════════════════
# 4. Regression trees — correctness
# ══════════════════════════════════════════════════════════════════════════════
header("Regression trees — correctness")

rng  = np.random.default_rng(7)
N, F = 120, 5
X    = rng.standard_normal((N, F))
Y    = rng.standard_normal(N)
X_l  = X.tolist()
Y_l  = Y.tolist()
idxs = list(range(N))

def leaf_cover(leaves):
    """All indexes covered exactly once."""
    covered = sorted(i for leaf in leaves for i in leaf[0])
    return covered

configs = [
    ("max_depth=3, min_ss=5",  dict(max_depth=3, min_samples_split=5)),
    ("max_depth=None, min_ss=10", dict(max_depth=None, min_samples_split=10)),
    ("max_depth=1, min_ss=2",  dict(max_depth=1, min_samples_split=2)),
]

for label, kw in configs:
    print(f"\n{BOLD}  {label}{RESET}")

    # ── Quadratic tree ─────────────────────────────────────────────────────────
    t = rs.RegressionTreeQuadratic(**kw)
    t.fit(X_l, Y_l)
    leaves = t.get_values_leaf(X_l, idxs)
    cover  = leaf_cover(leaves)
    ok(f"  RTQ: {len(leaves)} leaves, all {N} samples covered") \
        if cover == idxs else fail(f"  RTQ coverage mismatch {len(cover)} != {N}")

    if HAS_PYTHON:
        tp = PyRTQ(**kw)
        tp.fit(X, Y)
        py_leaves = tp.get_values_leaf(X, np.arange(N))
        py_cover  = sorted(leaf_cover(py_leaves))
        ok("  RTQ: Rust leaf split == Python leaf split") \
            if cover == py_cover else fail(f"  RTQ split differs: {set(cover)^set(py_cover)}")

    # ── CRPS tree ──────────────────────────────────────────────────────────────
    t = rs.RegressionTreeCRPS(**kw)
    t.fit(X_l, Y_l)
    leaves = t.get_values_leaf(X_l, idxs)
    cover  = leaf_cover(leaves)
    ok(f"  CRPS: {len(leaves)} leaves, all {N} samples covered") \
        if cover == idxs else fail(f"  CRPS coverage {len(cover)} != {N}")

    if HAS_PYTHON:
        tp = PyCRPS(**kw)
        tp.fit(X, Y)
        py_leaves = tp.get_values_leaf(X, np.arange(N))
        py_cover  = sorted(leaf_cover(py_leaves))
        ok("  CRPS: Rust leaf split == Python leaf split") \
            if cover == py_cover else fail(f"  CRPS split differs")

    # ── Quantile tree ──────────────────────────────────────────────────────────
    qs = [0.25, 0.5, 0.75]
    t  = rs.RegressionTreeQuantile(quantiles=qs, **kw)
    t.fit(X_l, Y_l)
    leaves = t.get_values_leaf(X_l, idxs)
    cover  = leaf_cover(leaves)
    ok(f"  QRT: {len(leaves)} leaves, all {N} samples covered") \
        if cover == idxs else fail(f"  QRT coverage {len(cover)} != {N}")

    if HAS_PYTHON:
        tp = PyQRT(quantiles=qs, **kw)
        tp.fit(X, Y)
        py_leaves = tp.get_values_leaf(X, np.arange(N))
        py_cover  = sorted(leaf_cover(py_leaves))
        ok("  QRT: Rust leaf split == Python leaf split") \
            if cover == py_cover else fail(f"  QRT split differs")

# y-values in leaves match
print(f"\n  {BOLD}Leaf y-values correctness{RESET}")
rng2 = np.random.default_rng(13)
X2   = rng2.standard_normal((60, 3))
Y2   = rng2.standard_normal(60)
kw2  = dict(max_depth=3, min_samples_split=4)

if HAS_PYTHON:
    for TreeRs, TreePy, name in [
        (rs.RegressionTreeQuadratic, PyRTQ, "RTQ"),
        (rs.RegressionTreeCRPS,      PyCRPS, "CRPS"),
    ]:
        t_rs = TreeRs(**kw2); t_rs.fit(X2.tolist(), Y2.tolist())
        t_py = TreePy(**kw2); t_py.fit(X2, Y2)
        rs_leaves = {tuple(sorted(l[0])): sorted(l[1]) for l in t_rs.get_values_leaf(X2.tolist(), list(range(60)))}
        py_leaves = {tuple(sorted(l[0])): sorted(l[1].tolist()) for l in t_py.get_values_leaf(X2, np.arange(60))}
        yvals_ok = all(
            abs(a - b) < 1e-12
            for key in rs_leaves
            for a, b in zip(rs_leaves.get(key, []), py_leaves.get(key, []))
        ) and set(rs_leaves.keys()) == set(py_leaves.keys())
        ok(f"  {name}: leaf y-values match Python exactly") if yvals_ok else fail(f"  {name}: y-value mismatch")


# ══════════════════════════════════════════════════════════════════════════════
# 5. Speed benchmarks
# ══════════════════════════════════════════════════════════════════════════════
if HAS_PYTHON:
    header("Speed benchmarks")

    rng_b = np.random.default_rng(99)

    print(f"\n{BOLD}  Entropy functions{RESET}")
    print(f"  {'Function':<38s}  {'Python':>9s}   {'Rust':>9s}   Speedup")
    print(f"  {'-'*38}  {'-'*9}   {'-'*9}   {'─'*10}")

    bench_cases = [
        ("entropies_quadratic  n=100",  100),
        ("entropies_quadratic  n=500",  500),
        ("entropies_quadratic  n=2000", 2000),
        ("entropies_crps       n=100",  100),
        ("entropies_crps       n=500",  500),
        ("entropies_crps       n=2000", 2000),
    ]
    for label, n in bench_cases:
        y_b = rng_b.standard_normal(n)
        o_b = np.argsort(rng_b.standard_normal(n))
        o_l = o_b.tolist(); y_l = y_b.tolist()
        fn   = "quadratic" if "quadratic" in label else "crps"
        nrep = max(3, min(50, 5000 // n))

        if fn == "quadratic":
            py_ms = timeit(lambda: py_ent_quad(o_b, y_b), nrep)
            rs_ms = timeit(lambda: rs.entropies_quadratic(o_l, y_l), nrep)
        else:
            py_ms = timeit(lambda: py_ent_crps(o_b, y_b), nrep)
            rs_ms = timeit(lambda: rs.entropies_crps(o_l, y_l), nrep)

        print(f"  {label:<38s}  {py_ms:>8.2f}ms  {rs_ms:>8.2f}ms  {speedup_str(py_ms, rs_ms)}")

    # Multi-quantile
    print()
    for qs, n in [([0.5], 200), ([0.25,0.5,0.75], 200), ([0.1,0.25,0.5,0.75,0.9], 200),
                  ([0.1,0.25,0.5,0.75,0.9], 500)]:
        y_b = rng_b.standard_normal(n)
        o_b = np.argsort(rng_b.standard_normal(n))
        o_l = o_b.tolist(); y_l = y_b.tolist()
        nrep = max(3, 1000 // n)
        label = f"entropies_multiq  q={len(qs)}  n={n}"
        py_ms = timeit(lambda: py_ent_mq(o_b, y_b, qs), nrep)
        rs_ms = timeit(lambda: rs.entropies_multi_quantiles(o_l, y_l, qs), nrep)
        print(f"  {label:<38s}  {py_ms:>8.2f}ms  {rs_ms:>8.2f}ms  {speedup_str(py_ms, rs_ms)}")

    print(f"\n{BOLD}  Tree fitting  (X: 200×6, depth=5, min_ss=5){RESET}")
    print(f"  {'Tree':<38s}  {'Python':>9s}   {'Rust':>9s}   Speedup")
    print(f"  {'-'*38}  {'-'*9}   {'-'*9}   {'─'*10}")

    X_bench = rng_b.standard_normal((200, 6))
    Y_bench = rng_b.standard_normal(200)
    X_bl = X_bench.tolist(); Y_bl = Y_bench.tolist()
    kw_bench = dict(max_depth=5, min_samples_split=5)
    NREP_TREE = 5

    # RTQ
    py_ms = timeit(lambda: (PyRTQ(**kw_bench).fit(X_bench, Y_bench)), NREP_TREE)
    rs_ms = timeit(lambda: (rs.RegressionTreeQuadratic(**kw_bench).fit(X_bl, Y_bl)), NREP_TREE)
    print(f"  {'RegressionTreeQuadratic':<38s}  {py_ms:>8.1f}ms  {rs_ms:>8.1f}ms  {speedup_str(py_ms, rs_ms)}")

    # CRPS
    py_ms = timeit(lambda: (PyCRPS(**kw_bench).fit(X_bench, Y_bench)), NREP_TREE)
    rs_ms = timeit(lambda: (rs.RegressionTreeCRPS(**kw_bench).fit(X_bl, Y_bl)), NREP_TREE)
    print(f"  {'RegressionTreeCRPS':<38s}  {py_ms:>8.1f}ms  {rs_ms:>8.1f}ms  {speedup_str(py_ms, rs_ms)}")

    # QRT
    qs5 = [0.1, 0.25, 0.5, 0.75, 0.9]
    py_ms = timeit(lambda: (PyQRT(quantiles=qs5, **kw_bench).fit(X_bench, Y_bench)), NREP_TREE)
    rs_ms = timeit(lambda: (rs.RegressionTreeQuantile(quantiles=qs5, **kw_bench).fit(X_bl, Y_bl)), NREP_TREE)
    print(f"  {'RegressionTreeQuantile (q=5)':<38s}  {py_ms:>8.1f}ms  {rs_ms:>8.1f}ms  {speedup_str(py_ms, rs_ms)}")

    # Larger dataset
    print(f"\n{BOLD}  Tree fitting  (X: 1000×10, depth=6, min_ss=5){RESET}")
    print(f"  {'Tree':<38s}  {'Python':>9s}   {'Rust':>9s}   Speedup")
    print(f"  {'-'*38}  {'-'*9}   {'-'*9}   {'─'*10}")
    X_big = rng_b.standard_normal((1000, 10))
    Y_big = rng_b.standard_normal(1000)
    X_bigl = X_big.tolist(); Y_bigl = Y_big.tolist()
    kw_big = dict(max_depth=6, min_samples_split=10)
    NREP_BIG = 3

    py_ms = timeit(lambda: (PyCRPS(**kw_big).fit(X_big, Y_big)), NREP_BIG)
    rs_ms = timeit(lambda: (rs.RegressionTreeCRPS(**kw_big).fit(X_bigl, Y_bigl)), NREP_BIG)
    print(f"  {'RegressionTreeCRPS':<38s}  {py_ms:>8.1f}ms  {rs_ms:>8.1f}ms  {speedup_str(py_ms, rs_ms)}")

    py_ms = timeit(lambda: (PyQRT(quantiles=qs5, **kw_big).fit(X_big, Y_big)), NREP_BIG)
    rs_ms = timeit(lambda: (rs.RegressionTreeQuantile(quantiles=qs5, **kw_big).fit(X_bigl, Y_bigl)), NREP_BIG)
    print(f"  {'RegressionTreeQuantile (q=5)':<38s}  {py_ms:>8.1f}ms  {rs_ms:>8.1f}ms  {speedup_str(py_ms, rs_ms)}")


# ══════════════════════════════════════════════════════════════════════════════
# 6. Summary
# ══════════════════════════════════════════════════════════════════════════════
header("Summary")
total_fail = len(_FAILURES)
if total_fail == 0:
    print(f"  {GREEN}{BOLD}All correctness checks passed!{RESET}")
else:
    print(f"  {RED}{BOLD}{total_fail} check(s) FAILED:{RESET}")
    for f in _FAILURES:
        print(f"    {RED}•{RESET} {f}")

if not HAS_PYTHON:
    print(f"\n  {YELLOW}⚠{RESET}  Python comparison skipped (DisTreebution not found).")
    print(f"  {DIM}Run with --py-path /path/to/DisTreebution/parent to enable it.{RESET}")

print()


────────────────────────────────────────────────────────────
  Importing packages
────────────────────────────────────────────────────────────
  ✓ distreebu_rs imported  (symbols: FenwickTree, MinMaxHeap, RegressionTreeCRPS, RegressionTreeQuadratic, RegressionTreeQuantile, distreebu_rs, entropies_crps, entropies_multi_quantiles, entropies_quadratic)
  ✓ DisTreebution (Python) imported

────────────────────────────────────────────────────────────
  Data structures
────────────────────────────────────────────────────────────

FenwickTree
  ✓ prefix_sum(0) == 0
  ✓ prefix_sum(1) == 1
  ✓ prefix_sum(3) == 6
  ✓ prefix_sum(5) == 15
  ✓ range_sum(1,3) == 5
  ✓ range_sum(2,5) == 11
  ✓ __getitem__(0) == 1
  ✓ __getitem__(3) == 5
  ✓ len == 5
  ✓ init() prefix_sum(5)==15

MinMaxHeap
  ✓ len == 5
  ✓ peekmin == 1.0
  ✓ peekmax == 5.0
  ✓ popmin == 1.0
  ✓ peekmin == 2.0
  ✓ popmax == 5.0
  ✓ peekmax == 4.0
  ✓ len == 3 after 2 pops
  ✓ MinMaxHeap stress: popmin gives sorted order (n=100)
  ✓ M

In [2]:
"""
train_crps_rf.py
================
Train a CRPS Random Forest on a synthetic dataset with 50 000 samples
and 400 features using the Rust-accelerated distreebu_rs backend.

The forest follows the same bagging protocol as DisTreebution's UQ.train_trees:
  - Each tree sees a 60 % bootstrap subsample (drawn without replacement).
  - At each node the full feature set is evaluated (no feature subsampling —
    matching the original implementation; add `max_features` below if desired).
  - Out-of-bag (OOB) indices are tracked per tree for downstream conformal use.

After training, the script:
  1. Routes each test point to its leaf in every tree.
  2. Pools the leaf y-values across trees and estimates the conditional CDF
     (distributional bagging / "vr-avg" aggregation).
  3. Computes empirical CRPS on the test set as a quality metric.
  4. Reports timing for every stage.

Usage
-----
    python train_crps_rf.py                # default synthetic data
    python train_crps_rf.py --n-trees 50  # fewer trees for a quick smoke-test
    python train_crps_rf.py --n-jobs 8    # parallel tree fitting (requires joblib)

Dependencies
------------
    pip install distreebu_rs-*.whl numpy joblib tqdm
"""

import argparse
import time
import sys
from typing import Dict, List, Tuple

import numpy as np

# ── CLI ────────────────────────────────────────────────────────────────────────
parser = argparse.ArgumentParser(description="CRPS Random Forest — Rust backend")
parser.add_argument("--n-samples",    type=int,   default=50_000,  help="Training set size")
parser.add_argument("--n-features",   type=int,   default=400,     help="Number of features")
parser.add_argument("--n-test",       type=int,   default=2_000,   help="Test set size")
parser.add_argument("--n-trees",      type=int,   default=100,     help="Number of trees in the forest")
parser.add_argument("--max-depth",    type=int,   default=6,       help="Max tree depth (None = unlimited)")
parser.add_argument("--min-ss",       type=int,   default=20,      help="min_samples_split per node")
parser.add_argument("--bag-fraction", type=float, default=0.6,     help="Fraction of data used per tree")
parser.add_argument("--n-jobs",       type=int,   default=1,       help="Parallel workers (1 = sequential)")
parser.add_argument("--seed",         type=int,   default=42,      help="Random seed")
parser.add_argument("--no-progress",  action="store_true",         help="Disable progress bar")
args = parser.parse_args()

# ── Imports ───────────────────────────────────────────────────────────────────
try:
    import distreebu_rs as rs
except ImportError:
    sys.exit("ERROR: distreebu_rs not found. Install with: pip install distreebu_rs-*.whl")

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

if args.n_jobs > 1:
    try:
        from joblib import Parallel, delayed
        HAS_JOBLIB = True
    except ImportError:
        print("WARNING: joblib not found — falling back to sequential (--n-jobs ignored)")
        HAS_JOBLIB = False
else:
    HAS_JOBLIB = False


# ══════════════════════════════════════════════════════════════════════════════
# 1.  Synthetic data
# ══════════════════════════════════════════════════════════════════════════════
def make_data(n: int, p: int, n_test: int, seed: int):
    """
    Heteroscedastic regression benchmark.

    y = sin(2π x₀) + x₁ · cos(π x₂) + 0.3 · (1 + |x₃|) · ε
    where ε ~ N(0,1).

    Only the first 4 features are informative; the remaining p-4 are pure noise.
    This lets us verify that the forest finds the relevant splits despite the
    large number of irrelevant features.
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n + n_test, p)).astype(np.float64)
    noise_scale = 0.3 * (1.0 + np.abs(X[:, 3]))
    y = (
        np.sin(2 * np.pi * X[:, 0])
        + X[:, 1] * np.cos(np.pi * X[:, 2])
        + noise_scale * rng.standard_normal(n + n_test)
    )
    return (
        X[:n],  y[:n],
        X[n:],  y[n:],
    )


# ══════════════════════════════════════════════════════════════════════════════
# 2.  Single-tree fitting (called in parallel or sequentially)
# ══════════════════════════════════════════════════════════════════════════════
def fit_one_tree(
    tree_id: int,
    X_train: np.ndarray,
    y_train: np.ndarray,
    bag_fraction: float,
    max_depth: int,
    min_ss: int,
    seed: int,
) -> Tuple[object, np.ndarray, np.ndarray]:
    """
    Bootstrap-sample the training set, fit one CRPS tree, return
    (tree, oob_indices, train_indices).
    """
    rng = np.random.default_rng(seed + tree_id)
    n = X_train.shape[0]
    n_bag = int(bag_fraction * n)

    # shuffle without replacement (same as DisTreebution's UQ.train_trees)
    perm = rng.permutation(n)
    idx_train = perm[:n_bag]
    idx_oob   = perm[n_bag:]

    X_bag = X_train[idx_train].tolist()
    y_bag = y_train[idx_train].tolist()

    tree = rs.RegressionTreeCRPS(max_depth=max_depth, min_samples_split=min_ss)
    tree.fit(X_bag, y_bag)

    return tree, idx_train, idx_oob


# ══════════════════════════════════════════════════════════════════════════════
# 3.  Prediction helpers
# ══════════════════════════════════════════════════════════════════════════════
def get_leaf_values(
    tree,
    X: np.ndarray,
    indexes: List[int],
) -> Dict[int, List[float]]:
    """
    Returns {sample_index: [y_values at leaf]} for every sample in `indexes`.
    """
    leaves = tree.get_values_leaf(X.tolist(), indexes)
    mapping: Dict[int, List[float]] = {}
    for leaf_idxs, leaf_yvals in leaves:
        for i in leaf_idxs:
            mapping[i] = list(leaf_yvals)
    return mapping


def forest_predict_quantiles(
    trees: List,
    X_train: np.ndarray,
    X_test: np.ndarray,
    quantiles: List[float],
) -> np.ndarray:
    """
    Distributional-bagging prediction ("vr-avg"):
      For each test point, pool all leaf y-values across trees with equal
      per-tree weight, then read off the desired quantiles.

    Returns array of shape (n_test, len(quantiles)).
    """
    n_test = X_test.shape[0]
    test_idxs = list(range(n_test))
    # Accumulate (value, weight) lists per test sample
    # weight = 1 / (leaf_size * n_trees)  → uniform across trees
    sample_pools: List[List[float]]   = [[] for _ in range(n_test)]
    sample_weights: List[List[float]] = [[] for _ in range(n_test)]

    for tree in trees:
        leaves = tree.get_values_leaf(X_test.tolist(), test_idxs)
        for leaf_idxs, leaf_yvals in leaves:
            leaf_size = len(leaf_yvals)
            w = 1.0 / leaf_size  # uniform within-leaf weight (will be re-normalised)
            for i in leaf_idxs:
                sample_pools[i].extend(leaf_yvals)
                sample_weights[i].extend([w] * leaf_size)

    out = np.empty((n_test, len(quantiles)))
    for i in range(n_test):
        vals = np.asarray(sample_pools[i])
        wts  = np.asarray(sample_weights[i])
        wts /= wts.sum()
        order = np.argsort(vals)
        vals_sorted = vals[order]
        cdf = np.cumsum(wts[order])
        for j, q in enumerate(quantiles):
            # inverse CDF: smallest value where CDF >= q
            idx = np.searchsorted(cdf, q)
            out[i, j] = vals_sorted[min(idx, len(vals_sorted) - 1)]
    return out


def empirical_crps(
    trees: List,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> float:
    """
    Energy-score form of CRPS:
      CRPS(F, y) = E|X - y| - 0.5 · E|X - X'|
    where X, X' ~ F (the empirical predictive CDF pooled across trees).

    Estimated via Monte Carlo with min(pool_size, 500) draws per sample.
    """
    n_test = X_test.shape[0]
    test_idxs = list(range(n_test))

    sample_pools: List[np.ndarray] = [[] for _ in range(n_test)]
    for tree in trees:
        leaves = tree.get_values_leaf(X_test.tolist(), test_idxs)
        for leaf_idxs, leaf_yvals in leaves:
            for i in leaf_idxs:
                sample_pools[i].extend(leaf_yvals)

    rng = np.random.default_rng(0)
    crps_vals = np.empty(n_test)
    for i in range(n_test):
        pool = np.asarray(sample_pools[i])
        if len(pool) > 500:
            pool = rng.choice(pool, 500, replace=False)
        # E|X - y|
        term1 = np.mean(np.abs(pool - y_test[i]))
        # 0.5 · E|X - X'|  via all pairs (or MC pairs)
        m = len(pool)
        if m > 200:
            idx_a = rng.integers(0, m, 2000)
            idx_b = rng.integers(0, m, 2000)
            term2 = 0.5 * np.mean(np.abs(pool[idx_a] - pool[idx_b]))
        else:
            diff = np.abs(pool[:, None] - pool[None, :])
            term2 = 0.5 * diff.mean()
        crps_vals[i] = term1 - term2

    return float(crps_vals.mean())


# ══════════════════════════════════════════════════════════════════════════════
# 4.  Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    SEP  = "─" * 62
    BOLD = "\033[1m"
    CYAN = "\033[96m"
    GRN  = "\033[92m"
    DIM  = "\033[2m"
    RST  = "\033[0m"

    def section(title):
        print(f"\n{BOLD}{CYAN}{SEP}{RST}")
        print(f"{BOLD}{CYAN}  {title}{RST}")
        print(f"{BOLD}{CYAN}{SEP}{RST}")

    def kv(k, v, unit=""):
        print(f"  {k:<30s} {BOLD}{v}{unit}{RST}")

    # ── Config ─────────────────────────────────────────────────────────────────
    section("Configuration")
    kv("n_samples (train)",   f"{args.n_samples:,}")
    kv("n_features",          f"{args.n_features:,}")
    kv("n_test",              f"{args.n_test:,}")
    kv("n_trees",             f"{args.n_trees}")
    kv("max_depth",           f"{args.max_depth}")
    kv("min_samples_split",   f"{args.min_ss}")
    kv("bag_fraction",        f"{args.bag_fraction:.0%}")
    kv("n_jobs",              f"{args.n_jobs}")
    kv("random_seed",         f"{args.seed}")
    mem_mb = args.n_samples * args.n_features * 8 / 1e6
    kv("X_train memory (float64)", f"{mem_mb:.0f}", " MB")

    # ── Data ───────────────────────────────────────────────────────────────────
    section("Generating synthetic data")
    t0 = time.perf_counter()
    X_train, y_train, X_test, y_test = make_data(
        args.n_samples, args.n_features, args.n_test, args.seed
    )
    kv("Data generated in", f"{time.perf_counter()-t0:.2f}", " s")
    kv("X_train shape", f"{X_train.shape}")
    kv("y_train  mean ± std",
       f"{y_train.mean():.3f} ± {y_train.std():.3f}")

    # ── Training ───────────────────────────────────────────────────────────────
    section(f"Training {args.n_trees} CRPS trees  (Rust backend)")
    t_train_start = time.perf_counter()

    # OOB tracking: sample → list of tree IDs in whose OOB it appears
    sample2oob: Dict[int, List[int]] = {i: [] for i in range(args.n_samples)}
    trees: List = [None] * args.n_trees

    if HAS_JOBLIB and args.n_jobs > 1:
        print(f"  {DIM}Parallel fitting with {args.n_jobs} workers (joblib)…{RST}")
        results = Parallel(n_jobs=args.n_jobs, backend="loky")(
            delayed(fit_one_tree)(
                tid, X_train, y_train,
                args.bag_fraction, args.max_depth, args.min_ss, args.seed,
            )
            for tid in range(args.n_trees)
        )
    else:
        results = []
        iterator = range(args.n_trees)
        if HAS_TQDM and not args.no_progress:
            iterator = tqdm(iterator, desc="  fitting trees", ncols=70,
                            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]")
        for tid in iterator:
            results.append(
                fit_one_tree(
                    tid, X_train, y_train,
                    args.bag_fraction, args.max_depth, args.min_ss, args.seed,
                )
            )

    for tid, (tree, idx_train, idx_oob) in enumerate(results):
        trees[tid] = tree
        for i in idx_oob:
            sample2oob[i].append(tid)

    t_train_total = time.perf_counter() - t_train_start
    kv("Total training time",       f"{t_train_total:.1f}", " s")
    kv("Average per tree",          f"{t_train_total/args.n_trees*1000:.0f}", " ms")
    kv("Throughput",
       f"{args.n_trees * int(args.bag_fraction * args.n_samples) / t_train_total / 1e6:.2f}",
       " M node-samples/s (approx)")

    oob_sizes = [len(v) for v in sample2oob.values()]
    kv("OOB trees/sample  mean±std",
       f"{np.mean(oob_sizes):.1f} ± {np.std(oob_sizes):.1f}")
    kv("Samples with ≥1 OOB tree",
       f"{sum(s > 0 for s in oob_sizes):,}")

    # ── Leaf statistics ────────────────────────────────────────────────────────
    section("Forest structure  (leaf statistics)")
    t0 = time.perf_counter()
    leaf_sizes_all = []
    for tid, tree in enumerate(trees):
        # count leaf sizes using the first tree's training indices as proxy
        _, idx_train, _ = results[tid]
        leaves_info = tree.get_values_leaf(
            X_train[idx_train[:min(1000, len(idx_train))]].tolist(),
            list(range(min(1000, len(idx_train)))),
        )
        for _, yvals in leaves_info:
            leaf_sizes_all.append(len(yvals))

    leaf_sizes_all = np.array(leaf_sizes_all)
    kv("Leaf size  mean",   f"{leaf_sizes_all.mean():.1f}")
    kv("Leaf size  median", f"{np.median(leaf_sizes_all):.0f}")
    kv("Leaf size  min/max",f"{leaf_sizes_all.min()} / {leaf_sizes_all.max()}")
    kv("Leaf stats time",   f"{time.perf_counter()-t0:.2f}", " s")

    # ── Prediction ─────────────────────────────────────────────────────────────
    section("Prediction on test set")
    quantiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    n_test_eval = min(500, args.n_test)
    print(f"  {DIM}Estimating quantiles {quantiles}{RST}")
    print(f"  {DIM}on {n_test_eval} test samples…{RST}")

    t0 = time.perf_counter()
    pred_q = forest_predict_quantiles(
        trees, X_train, X_test[:n_test_eval], quantiles
    )
    t_pred = time.perf_counter() - t0
    kv("Prediction time", f"{t_pred:.2f}", " s")
    kv("Throughput",
       f"{n_test_eval / t_pred:.0f}", " samples/s")

    # ── Quality metrics ────────────────────────────────────────────────────────
    section("Quality metrics")

    # Interval coverage and width for 80 % and 90 % nominal intervals
    for (lo_q, hi_q), nominal in [((0.10, 0.90), 0.80), ((0.05, 0.95), 0.90)]:
        lo_idx = quantiles.index(lo_q)
        hi_idx = quantiles.index(hi_q)
        lo_vals = pred_q[:, lo_idx]
        hi_vals = pred_q[:, hi_idx]
        coverage = np.mean(
            (y_test[:n_test_eval] >= lo_vals) & (y_test[:n_test_eval] <= hi_vals)
        )
        width = np.mean(hi_vals - lo_vals)
        kv(f"  {int(nominal*100)}% PI  coverage",
           f"{coverage:.3f}  (nominal {nominal:.2f})")
        kv(f"  {int(nominal*100)}% PI  avg width",
           f"{width:.3f}")

    # Median absolute error
    med_idx = quantiles.index(0.50)
    mae = np.mean(np.abs(pred_q[:, med_idx] - y_test[:n_test_eval]))
    kv("Median absolute error (at q=0.5)", f"{mae:.4f}")

    # CRPS (energy score, on a smaller subset for speed)
    n_crps = min(200, n_test_eval)
    print(f"\n  {DIM}Computing empirical CRPS on {n_crps} test samples…{RST}")
    t0 = time.perf_counter()
    crps_val = empirical_crps(trees, X_test[:n_crps], y_test[:n_crps])
    kv("Empirical CRPS  (mean)", f"{crps_val:.4f}", "")
    kv("CRPS eval time",         f"{time.perf_counter()-t0:.2f}", " s")

    # ── Summary ────────────────────────────────────────────────────────────────
    section("Summary")
    print(f"\n  {GRN}✓  Forest trained successfully{RST}")
    kv("Training time",          f"{t_train_total:.1f}", " s")
    kv("Prediction time (500 pts)", f"{t_pred:.2f}", " s")
    kv("Empirical CRPS",         f"{crps_val:.4f}")
    print(f"\n  {DIM}Trees are stored in the `trees` list — use get_values_leaf(){RST}")
    print(f"  {DIM}to route new samples and build distributional predictions.{RST}\n")

    return trees, sample2oob


if __name__ == "__main__":
    trees, sample2oob = main()

usage: ipykernel_launcher.py [-h] [--n-samples N_SAMPLES]
                             [--n-features N_FEATURES] [--n-test N_TEST]
                             [--n-trees N_TREES] [--max-depth MAX_DEPTH]
                             [--min-ss MIN_SS] [--bag-fraction BAG_FRACTION]
                             [--n-jobs N_JOBS] [--seed SEED] [--no-progress]
ipykernel_launcher.py: error: unrecognized arguments: -f /myhome/.local/share/jupyter/runtime/kernel-69e5c014-1c92-49f3-89e0-d2f41a85bd5d.json


SystemExit: 2

/opt/conda/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
